In [4]:
%matplotlib inline
import matplotlib.pyplot as plt
import IPython.display as ipd

import os
import json
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioLoader, TextAudioCollate, TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence

from scipy.io.wavfile import write


def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=1023)
           2	RESUME(arg=0, lineno=1023)
           4	LOAD_FAST(arg=0, lineno=1026)
           6	LOAD_CONST(arg=1, lineno=1026)
           8	BINARY_SUBSCR(arg=None, lineno=1026)
          12	LOAD_FAST(arg=0, lineno=1026)
          14	LOAD_CONST(arg=2, lineno=1026)
          16	BINARY_SUBSCR(arg=None, lineno=1026)
          20	COMPARE_OP(arg=68, lineno=1026)
          24	LOAD_FAST(arg=0, lineno=1026)
          26	LOAD_CONST(arg=1, lineno=1026)
          28	BINARY_SUBSCR(arg=None, lineno=1026)
          32	LOAD_FAST(arg=0, lineno=1026)
          34	LOAD_CONST(arg=3, lineno=1026)
          36	BINARY_SUBSCR(arg=None, lineno=1026)
          40	COMPARE_OP(arg=92, lineno=1026)
          44	BINARY_OP(arg=1, lineno=1026)
          48	RETURN_VALUE(arg=None, lineno=1026)
DEBUG:numba.core.byteflow:pending: deque([State(pc_initial=0 nstack_initial=0)])
DEBUG:numba.core.byteflow:stack: []
DEBUG:numba.core.byteflow:state.

In [5]:
from phonemizer.backend.espeak.wrapper import EspeakWrapper
_ESPEAK_LIBRARY = '/opt/homebrew/Cellar/espeak-ng/1.52.0/lib/libespeak-ng.dylib'  #use the Path to the library.
EspeakWrapper.set_library(_ESPEAK_LIBRARY)

## LJ Speech

In [6]:
hps = utils.get_hparams_from_file("./configs/ljs_base.json")

In [7]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model)
_ = net_g.eval()

_ = utils.load_checkpoint("/Users/cxhui/PycharmProjects/AudioProcess/Projects/vits/checkpoints/pretrained_ljs.pth", net_g, None)

/opt/homebrew/Caskroom/miniconda/base/envs/vits/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


INFO:root:Loaded checkpoint '/Users/cxhui/PycharmProjects/AudioProcess/Projects/vits/checkpoints/pretrained_ljs.pth' (iteration 0)


In [9]:
stn_tst = get_text("VITS is Awesome!", hps)
with torch.no_grad():
    x_tst = stn_tst.unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)])
    audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

## VCTK

In [10]:
hps = utils.get_hparams_from_file("./configs/vctk_base.json")

In [11]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    n_speakers=hps.data.n_speakers,
    **hps.model)
_ = net_g.eval()

_ = utils.load_checkpoint("/Users/cxhui/PycharmProjects/AudioProcess/Projects/vits/checkpoints/pretrained_vctk.pth", net_g, None)

INFO:root:Loaded checkpoint '/Users/cxhui/PycharmProjects/AudioProcess/Projects/vits/checkpoints/pretrained_vctk.pth' (iteration 0)


In [13]:
stn_tst = get_text("VITS is Awesome!", hps)
with torch.no_grad():
    x_tst = stn_tst.unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)])
    sid = torch.LongTensor([4])
    audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

### Voice Conversion

In [14]:
dataset = TextAudioSpeakerLoader(hps.data.validation_files, hps.data)
collate_fn = TextAudioSpeakerCollate()
loader = DataLoader(dataset, num_workers=8, shuffle=False,
    batch_size=1, pin_memory=True,
    drop_last=True, collate_fn=collate_fn)
data_list = list(loader)

FileNotFoundError: [Errno 2] No such file or directory: 'DUMMY2/p234/p234_071.wav'

In [ ]:
with torch.no_grad():
    x, x_lengths, spec, spec_lengths, y, y_lengths, sid_src = [x for x in data_list[0]]
    sid_tgt1 = torch.LongTensor([1])
    sid_tgt2 = torch.LongTensor([2])
    sid_tgt3 = torch.LongTensor([4])
    audio1 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt1)[0][0,0].data.cpu().float().numpy()
    audio2 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt2)[0][0,0].data.cpu().float().numpy()
    audio3 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt3)[0][0,0].data.cpu().float().numpy()
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].cpu().numpy(), rate=hps.data.sampling_rate, normalize=False))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=hps.data.sampling_rate, normalize=False))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=hps.data.sampling_rate, normalize=False))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=hps.data.sampling_rate, normalize=False))